# CADI Deep Learning - Semana 9
# # Semana 9:  Implementación de Data Augmentation y Transfer Learning en Imágenes en Google Colab
# **Estudiante:** Jenifer Cárdenas Aguilera

In [20]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt
import numpy as np

# 1. Carga y Preparación de Datos (Fashion-MNIST)
# Usamos este dataset para asegurar estabilidad en la descarga
(imagenes_entrenamiento, etiquetas_entrenamiento), (imagenes_test, etiquetas_test) = datasets.fashion_mnist.load_data()

# Convertimos a 3 canales (RGB) para compatibilidad con MobileNetV2
imagenes_entrenamiento = np.repeat(imagenes_entrenamiento[..., np.newaxis], 3, -1)
imagenes_test = np.repeat(imagenes_test[..., np.newaxis], 3, -1)

# Redimensionar a 32x32 (requisito mínimo del modelo preentrenado)
imagenes_entrenamiento = tf.image.resize(imagenes_entrenamiento, [32, 32]).numpy()
imagenes_test = tf.image.resize(imagenes_test, [32, 32]).numpy()

# Normalización de píxeles
imagenes_entrenamiento, imagenes_test = imagenes_entrenamiento / 255.0, imagenes_test / 255.0

nombres_clases = ['Camiseta', 'Pantalón', 'Suéter', 'Vestido', 'Abrigo', 'Sandalia', 'Camisa', 'Zapatilla', 'Bolso', 'Botín']
print('¡Datos de Fashion-MNIST cargados y personalizados con éxito!')

¡Datos de Fashion-MNIST cargados y personalizados con éxito!


In [ ]:
# 2. Implementación de Data Augmentation
data_augmentation = tf.keras.Sequential([
  layers.RandomFlip("horizontal"),
  layers.RandomRotation(0.1),
  layers.RandomZoom(0.1),
])

# Visualización
plt.figure(figsize=(10, 4))
for i in range(4):
  # Usamos el nuevo nombre de la variable personalizada
  augmented_image = data_augmentation(np.expand_dims(imagenes_entrenamiento[0], 0))
  plt.subplot(1, 4, i + 1)
  plt.imshow(augmented_image[0])
  plt.title(f"Variación {i+1}")
  plt.axis("off")
plt.suptitle("Ejemplo de Data Augmentation (Misma imagen transformada)")
plt.show()

In [ ]:
# Reducimos el set de datos para que el entrenamiento sea veloz en la clase
fotos_entreno_peque = imagenes_entrenamiento[:5000]
etiquetas_entreno_peque = etiquetas_entrenamiento[:5000]

# --- MODELO 1: ARQUITECTURA BASICA ---
red_basica = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(10)
])
red_basica.compile(optimizer='adam', loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])

# Elegi 3 epocas para no demorar la presentacion y evitar que la GPU de Colab se sature
print('Entrenando Red Basica...')
h_base = red_basica.fit(fotos_entreno_peque, etiquetas_entreno_peque, epochs=3, validation_split=0.2, verbose=1)

# --- MODELO 2: RED CON DATA AUGMENTATION ---
red_con_aumento = models.Sequential([
    data_augmentation,
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(10)
])
red_con_aumento.compile(optimizer='adam', loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
print('\nEntrenando Red con Aumento de Datos...')
h_aug = red_con_aumento.fit(fotos_entreno_peque, etiquetas_entreno_peque, epochs=3, validation_split=0.2, verbose=1)

# --- MODELO 3: RED EXPERTA (TRANSFER LEARNING) ---
# Elegi MobileNetV2 porque es una arquitectura optimizada para dispositivos moviles.
# Esto la hace ideal para nuestro proyecto ya que es muy rapida, ocupa poca memoria
# y ya conoce millones de caracteristicas visuales gracias a ImageNet.
base_mobilenet = tf.keras.applications.MobileNetV2(input_shape=(32, 32, 3), include_top=False, weights='imagenet')
base_mobilenet.trainable = False

red_experta = models.Sequential([
    base_mobilenet,
    layers.GlobalAveragePooling2D(),
    layers.Dense(10)
])
red_experta.compile(optimizer='adam', loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
print('\nEntrenando Red Experta (Transfer Learning)...')
h_tl = red_experta.fit(fotos_entreno_peque, etiquetas_entreno_peque, epochs=3, validation_split=0.2, verbose=1)

In [ ]:
# 3. Comparación de resultados
plt.figure(figsize=(10, 6))
plt.plot(h_base.history['val_accuracy'], label='Modelo Base', marker='o')
plt.plot(h_aug.history['val_accuracy'], label='Con Augmentation', marker='s')
plt.plot(h_tl.history['val_accuracy'], label='Transfer Learning', marker='^')
plt.title('Comparativa de Precisión (Validation Accuracy)')
plt.xlabel('Época')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

## Análisis y Conclusiones Finales

### 1. Modelo Base
El modelo base es una arquitectura sencilla. Su principal limitación es que tiende al **sobreajuste (overfitting)** rápidamente, ya que solo conoce un número limitado de imágenes fijas.

### 2. Impacto de Data Augmentation
Al aplicar transformaciones aleatorias, el modelo nunca ve la misma imagen exacta dos veces. Esto obliga a la red a aprender características más generales (como la forma de un objeto) en lugar de memorizar píxeles específicos, mejorando la **generalización**.

### 3. Ventajas de Transfer Learning
Utilizar **MobileNetV2** nos permite aprovechar el conocimiento de un modelo entrenado con millones de imágenes. Es la técnica más eficiente porque:
- Requiere menos tiempo de entrenamiento.
- Logra mayor precisión con menos datos.
- Es ideal para aplicaciones móviles o dispositivos con pocos recursos.

**Conclusión:** Para proyectos escolares o profesionales con pocos datos, el Transfer Learning combinado con Data Augmentation suele ser la mejor estrategia para obtener resultados rápidos y precisos.